In [ ]:
import h5py
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.optimize import curve_fit
from matplotlib.patches import Rectangle
from signals.PeakSignal import PeakSignal
from skimage.transform import rotate, radon
from tools import analyse_waveforms, save_analysis, read_analysis, H5FileManager

In [ ]:
sample_name = "test"
wafer_type = "test"
data_folder = "/path/to/data"

In [ ]:
# voltages = []

In [ ]:
# for voltage in voltages:
#     df = analyse_waveforms(
#         f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}.h5", voltage
#     )
#     save_analysis(
#         df, f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}_{voltage}V.csv"
#     )

In [ ]:
# save_analysis(df, f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}_{voltage}V.csv")

In [ ]:
# H5FileManager.split_by_voltage(f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}.h5")
# H5FileManager.merge_files(f"{data_folder}/{wafer_type}/{sample_name}/combined_results.h5", files_to_combine)

In [ ]:
def sigmoid(x, amp, cen, wid, const):
    return amp / (1 + np.exp(-1 * (x - cen) / wid)) + const


def sigmoid_inverse(x, amp, cen, wid, const):
    return cen - wid * np.log(amp / (x - const) - 1)


def gauss(x, amp, cen, wid):
    return amp * np.exp(-((x - cen) ** 2) / 2 / wid / wid)

In [ ]:
def plot_amplitude_map(df, voltage, x_borders=None, y_borders=None):
    agg_df = (
        df[df["pulse_number"] == 1]
        .groupby(["x", "y"], as_index=False)["peak_amplitude"]
        .mean()
    )
    pivot = agg_df.pivot(index="y", columns="x", values="peak_amplitude")
    pivot = pivot.sort_index().sort_index(axis=1)

    _, ax = plt.subplots(figsize=(10, 8))

    sns.heatmap(
        np.abs(pivot),
        cmap="plasma",
        cbar_kws={"label": "Mean amplitude [V]"},
        ax=ax,
    )

    ax.set_title(f"TCT Scan - {voltage} V - Combined Channels")
    ax.set_xlabel(r"x ($\mu$m)")
    ax.set_ylabel(r"y ($\mu$m)")

    if x_borders is not None and y_borders is not None:

        x_vals = pivot.columns.values
        y_vals = pivot.index.values

        for key in x_borders.keys():

            xb = x_borders[key]
            yb = y_borders[key]

            x0 = np.searchsorted(x_vals, xb[0])
            x1 = np.searchsorted(x_vals, xb[1])
            y0 = np.searchsorted(y_vals, yb[0])
            y1 = np.searchsorted(y_vals, yb[1])

            rect = Rectangle(
                (x0, y0),
                x1 - x0,
                y1 - y0,
                linewidth=2,
                edgecolor="black",
                facecolor="none",
            )

            ax.add_patch(rect)

    plt.tight_layout()
    plt.show()

In [ ]:
voltage = 130

In [ ]:
df = read_analysis(
    f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}_{voltage}V.csv"
)

In [ ]:
plot_amplitude_map(df, voltage)

In [ ]:
def get_corrected_df(df):
    pivot_for_angle = (
        df[df["pulse_number"] == 1]
        .pivot_table(index="y", columns="x", values="peak_amplitude", aggfunc="mean")
        .fillna(0)
    )
    image = np.abs(pivot_for_angle.values)
    theta = np.linspace(0.0, 180.0, 180, endpoint=False)
    sinogram = radon(image, theta=theta, circle=False)
    best_angle_deg = theta[np.argmax(np.var(sinogram, axis=0))]

    x_orig_coords = np.sort(df["x"].unique())
    y_orig_coords = np.sort(df["y"].unique())
    dx = np.median(np.diff(x_orig_coords))
    dy = np.median(np.diff(y_orig_coords))
    x_mid, y_mid = x_orig_coords.mean(), y_orig_coords.mean()

    prefixes = ("peak_time", "peak_amplitude", "peak_integral")
    cols_to_rotate = [c for c in df.columns if c.startswith(prefixes)]

    corrected_chunks = []

    for ch in df["ch"].unique():
        df_ch = df[(df["ch"] == ch) & (df["pulse_number"] == 1)]
        if df_ch.empty:
            continue

        rotated_data = {}
        h_new, w_new = 0, 0

        for col in cols_to_rotate:
            if col not in df_ch.columns:
                continue

            pivot_col = df_ch.pivot_table(
                index="y", columns="x", values=col, aggfunc="mean"
            ).fillna(0)

            rotated_img = rotate(
                pivot_col.values,
                -best_angle_deg,
                resize=True,
                order=3,
                mode="constant",
                cval=0,
            )
            rotated_data[col] = rotated_img.flatten()
            h_new, w_new = rotated_img.shape

        xi = np.linspace(x_mid - (w_new * dx) / 2, x_mid + (w_new * dx) / 2, w_new)
        yi = np.linspace(y_mid - (h_new * dy) / 2, y_mid + (h_new * dy) / 2, h_new)
        xi_grid, yi_grid = np.meshgrid(xi, yi)

        chunk = pd.DataFrame(
            {
                "x": xi_grid.flatten(),
                "y": yi_grid.flatten(),
                "ch": ch,
                "pulse_number": 1,
            }
        )

        for col, values in rotated_data.items():
            chunk[col] = values

        corrected_chunks.append(chunk)

    df_final = pd.concat(corrected_chunks, ignore_index=True)
    df_final["x"] = df_final["x"].round(1)
    df_final["y"] = df_final["y"].round(1)

    print(f"Sensor aligned! Angle: {-best_angle_deg:.2f}°")
    return df_final

In [ ]:
df_corrected = get_corrected_df(df)

In [ ]:
plot_amplitude_map(df_corrected, voltage)

In [ ]:
def plot_y_profile(df, voltage, threshold=0.9, make_plot=False):
    y_borders = defaultdict(list)
    for ch in sorted(df["ch"].unique()):
        df_ch = df[(df["ch"] == ch) & (df["pulse_number"] == 1)]

        agg_df = df_ch.groupby("y", as_index=False)["peak_amplitude"].sum()
        ylabel = "Sum of mean amplitudes [V]"

        agg_df = agg_df.sort_values("y")
        if make_plot:
            plt.plot(
                agg_df["y"],
                np.abs(agg_df["peak_amplitude"]),
                marker="o",
                label=f"Channel {ch}",
            )

        above_threshold = agg_df[
            np.abs(agg_df["peak_amplitude"])
            > threshold * np.max(np.abs(agg_df["peak_amplitude"]))
        ]

        x_left = agg_df[
            (agg_df["y"] >= agg_df["y"].min())
            & (
                agg_df["y"]
                <= 0.5 * (above_threshold["y"].min() + above_threshold["y"].max())
            )
        ]["y"].values
        y_left = np.abs(
            agg_df[
                (agg_df["y"] >= agg_df["y"].min())
                & (
                    agg_df["y"]
                    <= 0.5 * (above_threshold["y"].min() + above_threshold["y"].max())
                )
            ]["peak_amplitude"].values
        )

        x_right = agg_df[
            (
                agg_df["y"]
                >= 0.5 * (above_threshold["y"].min() + above_threshold["y"].max())
            )
            & (agg_df["y"] <= agg_df["y"].max())
        ]["y"].values
        y_right = np.abs(
            agg_df[
                (
                    agg_df["y"]
                    >= 0.5 * (above_threshold["y"].min() + above_threshold["y"].max())
                )
                & (agg_df["y"] <= agg_df["y"].max())
            ]["peak_amplitude"].values
        )

        popt, _ = curve_fit(
            sigmoid,
            x_left,
            y_left,
            p0=[np.max(y_left), x_left.mean(), 10, 0],
            maxfev=10000,
        )

        if make_plot:
            plt.axvline(
                sigmoid_inverse(threshold * popt[0], *popt),
                color="black",
                linestyle="--",
            )
        y_borders[f"ch_{ch}"].append(sigmoid_inverse(threshold * popt[0], *popt))
        popt, _ = curve_fit(
            sigmoid,
            x_right,
            y_right,
            p0=[np.max(y_right), x_right.mean(), -10, 0],
            maxfev=10000,
        )

        if make_plot:
            plt.axvline(
                sigmoid_inverse(threshold * popt[0], *popt),
                color="black",
                linestyle="--",
            )
        y_borders[f"ch_{ch}"].append(sigmoid_inverse(threshold * popt[0], *popt))

    min_key = min(y_borders, key=lambda k: min(y_borders[k]))
    max_key = max(y_borders, key=lambda k: max(y_borders[k]))

    interpad_distance = min(y_borders[max_key]) - max(y_borders[min_key])
    y_borders["interpad"] = [min(y_borders[max_key]), max(y_borders[min_key])]

    if make_plot:
        plt.title(f"TCT Scan - {voltage} V - Y Profile")
        plt.xlabel(r"y ($\mu$m)")
        plt.ylabel(ylabel)
        plt.text(
            0.5,
            0.15,
            f"Inter-pad distance: {interpad_distance:.1f} μm, {threshold*100:.0f}% threshold",
            ha="center",
            va="center",
            transform=plt.gca().transAxes,
            fontsize=12,
            bbox=dict(facecolor="white", alpha=0.8),
        )
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    return y_borders, interpad_distance

In [ ]:
y_borders, interpad_distance = plot_y_profile(
    df_corrected, voltage, threshold=0.9, make_plot=True
)

In [ ]:
def plot_x_profile(df, voltage, threshold=0.9, make_plot=False):
    x_borders = defaultdict(list)
    for ch in sorted(df["ch"].unique()):

        df_ch = df[(df["ch"] == ch) & (df["pulse_number"] == 1)]
        agg_df = df_ch.groupby("x", as_index=False)["peak_amplitude"].sum()
        xlabel = "Sum of mean amplitudes [V]"

        agg_df = agg_df.sort_values("x")
        agg_df = agg_df[agg_df["x"] != 0]

        if make_plot:
            plt.plot(
                agg_df["x"],
                np.abs(agg_df["peak_amplitude"]),
                marker="o",
                label=f"Channel {ch}",
            )

        x_left = agg_df[agg_df["x"] <= agg_df["x"].mean()]["x"].values
        y_left = np.abs(
            agg_df[agg_df["x"] <= agg_df["x"].median()]["peak_amplitude"].values
        )

        x_right = agg_df[agg_df["x"] > agg_df["x"].median()]["x"].values
        y_right = np.abs(
            agg_df[agg_df["x"] > agg_df["x"].mean()]["peak_amplitude"].values
        )

        popt, _ = curve_fit(
            sigmoid,
            x_left,
            y_left,
            p0=[np.max(y_left), x_left.mean(), 10, 0],
            maxfev=10000,
        )

        x_borders[f"ch_{ch}"].append(sigmoid_inverse(threshold * popt[0], *popt))

        popt, _ = curve_fit(
            sigmoid,
            x_right,
            y_right,
            p0=[np.max(y_right), x_right.mean(), -10, 50],
            maxfev=10000,
        )

        x_borders[f"ch_{ch}"].append(sigmoid_inverse(threshold * popt[0], *popt))

        min_ch_num = min(int(k.split("_")[1]) for k in x_borders.keys())

    for ch in sorted(df["ch"].unique()):
        val1, val2 = x_borders[f"ch_{ch}"]
        val3, val4 = x_borders[f"ch_{ch}"]
        x_borders["interpad"] = [0.5 * (val1 + val3), 0.5 * (val2 + val4)]

        if make_plot:
            if ch == min_ch_num:
                plt.axvline(min(x_borders[f"ch_{ch}"]), linestyle="--")
                plt.axvline(max(x_borders[f"ch_{ch}"]), linestyle="--")
            else:
                plt.axvline(
                    min(x_borders[f"ch_{ch}"]),
                    color="orange",
                    linestyle="--",
                )
                plt.axvline(
                    max(x_borders[f"ch_{ch}"]),
                    color="orange",
                    linestyle="--",
                )
    if make_plot:
        plt.title(f"TCT Scan - {voltage} V - X Profile")
        plt.xlabel(r"x ($\mu$m)")
        plt.ylabel(xlabel)
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    return x_borders

In [ ]:
x_borders = plot_x_profile(df_corrected, voltage, threshold=0.4, make_plot=True)

In [ ]:
plot_amplitude_map(df_corrected, voltage, x_borders=x_borders, y_borders=y_borders)

In [ ]:
def plot_jitter(
    df,
    percentage,
    x_borders=None,
    y_borders=None,
    fit_options={
        "bins": 100,
        "left_lim": 98.3,
        "right_lim": 99.0,
        "peak_left": 98.45,
        "peak_right": 98.9,
    },
    make_plot=False,
):
    bins = fit_options["bins"]
    left_lim, right_lim = fit_options["left_lim"], fit_options["right_lim"]
    peak_left, peak_right = fit_options["peak_left"], fit_options["peak_right"]

    if make_plot:
        plt.figure(figsize=(10, 6))

    jitters = defaultdict(list)
    all_jitter_values = []

    for ch in sorted(df["ch"].unique()):
        df_ch = df[df["ch"] == ch]

        if x_borders[f"ch_{ch}"] is not None:
            df_ch = df_ch[
                (df_ch["x"] >= min(x_borders[f"ch_{ch}"]))
                & (df_ch["x"] <= max(x_borders[f"ch_{ch}"]))
            ]
        if y_borders[f"ch{ch}"] is not None:
            df_ch = df_ch[
                (df_ch["y"] >= min(y_borders[f"ch_{ch}"]))
                & (df_ch["y"] <= max(y_borders[f"ch_{ch}"]))
            ]

        pivot = df_ch.pivot_table(
            index=["x", "y", "z", "waveform_index"],
            columns="pulse_number",
            values=f"peak_time_{percentage}",
        )

        pivot = pivot.dropna(subset=[1, 2])

        jitter = pivot[2] - pivot[1]
        all_jitter_values.extend(jitter.values)

        y, bin_edges = np.histogram(
            jitter.values,
            bins=int(bins / (right_lim - left_lim) * (peak_right - peak_left)),
            range=(peak_left, peak_right),
        )
        x = (bin_edges[:-1] + bin_edges[1:]) / 2

        popt, _ = curve_fit(
            gauss,
            x,
            y,
            p0=[max(y), np.mean(jitter.values), np.std(jitter.values)],
            maxfev=10000,
            bounds=(
                [0, fit_options["peak_left"], 0],
                [np.inf, fit_options["peak_right"], np.inf],
            ),
        )
        std_dev = popt[2]
        jitters[f"ch_{ch}"] = std_dev
        if make_plot:
            x = np.linspace(x[0], x[-1], 1000)
            plt.hist(
                jitter.values,
                bins=bins,
                range=(left_lim, right_lim),
                alpha=0.5,
                label=rf"Jitter Ch {ch} ($\sigma$={1e3*std_dev:.3f} ps)",
                zorder=1,
            )
            plt.plot(
                x,
                gauss(x, *popt),
                "k-",
                linewidth=2,
            )
            plt.xlabel("Pulses delay, [ns]")
            plt.legend()

    if x_borders[f"interpad"] is not None:
        df_interpad = df[
            (df["x"] >= min(x_borders[f"interpad"]))
            & (df["x"] <= max(x_borders[f"interpad"]))
        ]

    if y_borders[f"interpad"] is not None:
        df_interpad = df[
            (df["y"] >= min(y_borders[f"interpad"]))
            & (df["y"] <= max(y_borders[f"interpad"]))
        ]

    pivot = df_interpad.pivot_table(
        index=["x", "y", "z", "waveform_index"],
        columns="pulse_number",
        values=f"peak_time_{percentage}",
    )

    pivot = pivot.dropna(subset=[1, 2])

    jitter = pivot[2] - pivot[1]
    all_jitter_values.extend(jitter.values)

    y, bin_edges = np.histogram(
        jitter.values,
        bins=int(bins / (right_lim - left_lim) * (peak_right - peak_left)),
        range=(peak_left, peak_right),
    )
    x = (bin_edges[:-1] + bin_edges[1:]) / 2

    popt, _ = curve_fit(
        gauss,
        x,
        y,
        p0=[max(y), np.mean(jitter.values), np.std(jitter.values)],
        maxfev=10000,
        bounds=(
            [0, fit_options["peak_left"], 0],
            [np.inf, fit_options["peak_right"], np.inf],
        ),
    )
    std_dev = popt[2]
    jitters[f"interpad"] = std_dev

    y, bin_edges = np.histogram(
        all_jitter_values,
        bins=int(bins / (right_lim - left_lim) * (peak_right - peak_left)),
        range=(peak_left, peak_right),
    )
    x = (bin_edges[:-1] + bin_edges[1:]) / 2

    popt_full, _ = curve_fit(
        gauss,
        x,
        y,
        p0=[max(y), np.mean(all_jitter_values), np.std(all_jitter_values)],
        maxfev=10000,
        bounds=(
            [0, fit_options["peak_left"], 0],
            [np.inf, fit_options["peak_right"], np.inf],
        ),
    )
    std_dev_full = popt_full[2]
    jitters[f"full"] = std_dev_full

    if make_plot:
        x = np.linspace(x[0], x[-1], 1000)
        plt.hist(
            jitter.values,
            bins=bins,
            range=(left_lim, right_lim),
            alpha=0.5,
            label=rf"Jitter inter-pad ($\sigma$={1e3*std_dev:.3f} ps)",
            zorder=2,
        )
        plt.plot(
            x,
            gauss(x, *popt),
            "k-",
            linewidth=2,
        )
        plt.hist(
            all_jitter_values,
            bins=bins,
            range=(left_lim, right_lim),
            alpha=0.5,
            label=rf"Jitter full ($\sigma$={1e3*std_dev_full:.3f} ps)",
            zorder=0,
        )
        plt.plot(
            x,
            gauss(x, *popt_full),
            "k-",
            linewidth=2,
        )
        plt.legend()

    return jitters

In [ ]:
percentage = 20
jitters = plot_jitter(
    df,
    percentage,
    x_borders=x_borders,
    y_borders=y_borders,
    fit_options={
        "bins": 100,
        "left_lim": 98.5,
        "right_lim": 98.8,
        "peak_left": 98.5,
        "peak_right": 98.8,
    },
    make_plot=True,
)

In [ ]:
def plot_waveform(sample_path, voltage, position, channel, index=0):
    target_pos = np.array(position)

    with h5py.File(sample_path, "r") as f:
        grp_name = f"voltage_{voltage:04d}V"
        if grp_name not in f:
            print(f"Error: Group {grp_name} not found in HDF5 file.")
            return

        voltage_grp = f[grp_name]
        closest_key = None
        closest_pos = None
        min_dist = float("inf")

        for key in voltage_grp.keys():
            subgroup = voltage_grp[key]

            if all(k in subgroup.attrs for k in ("x", "y", "z")):
                curr_x = 1e6 * subgroup.attrs["x"]
                curr_y = 1e6 * subgroup.attrs["y"]
                curr_z = 1e6 * subgroup.attrs["z"]

                current_pos = np.array([curr_x, curr_y, curr_z])

                dist = np.linalg.norm(current_pos - target_pos)

                if dist < min_dist:
                    min_dist = dist
                    closest_key = key
                    closest_pos = current_pos

        if closest_key is None:
            print("Could not find any groups with x, y, z attributes.")
            return

        if min_dist == 0:
            print(f"Found exact match: {closest_pos} in group '{closest_key}'")
        else:
            print(f"Position {position} not found.")
            print(
                f"Closest match: {closest_pos} in group '{closest_key}' (Dist: {min_dist:.2f})"
            )

        pos_grp = voltage_grp[closest_key]

        ds_name = f"ch{channel}_v"

        ch_data = pos_grp[ds_name]

        dt = ch_data.attrs["dt"]
        t0 = ch_data.attrs["t0"]

        n_samples = ch_data.shape[1]
        half = n_samples // 2

        t = t0 + np.arange(n_samples) * dt
        t *= 1e9

        waveforms = ch_data[:][index]
        plt.plot(t[:half], waveforms[:half], label="Pulse 1")
        plt.plot(t[half:], waveforms[half:], label="Pulse 2")
        plt.xlabel("t, [ns]")
        plt.ylabel("Amplitude, [V]")
        plt.title(
            f"{voltage} V, ch={channel} x={position[0]} y={position[1]} z={position[2]}, wf_idx={index}"
        )
        plt.legend()

In [ ]:
position, ch = [-3367, -1735, 68000], 3
plot_waveform(
    f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}.h5",
    voltage,
    position,
    ch,
    0,
)

In [ ]:
def plot_mean_charge_map(df, voltage, x_borders, y_borders):
    agg_df = (
        df[df["pulse_number"] == 1]
        .groupby(["x", "y"], as_index=False)["peak_integral"]
        .mean()
    )

    mean_charges = {}

    pivot = agg_df.pivot(index="y", columns="x", values="peak_integral")
    pivot = pivot.sort_index().sort_index(axis=1)

    _, ax = plt.subplots(figsize=(10, 8))

    sns.heatmap(
        np.abs(pivot),
        cmap="plasma",
        cbar_kws={"label": r"Total sum of impedance-charge [V$\cdot$ns]"},
        ax=ax,
    )

    ax.set_title(f"TCT Scan - {voltage} V - Combined Channels")
    ax.set_xlabel(r"x ($\mu$m)")
    ax.set_ylabel(r"y ($\mu$m)")

    x_vals = pivot.columns.values
    y_vals = pivot.index.values

    for key in x_borders.keys():
        xb = x_borders[key]
        yb = y_borders[key]

        mask = (
            (agg_df["x"] >= min(xb))
            & (agg_df["x"] <= max(xb))
            & (agg_df["y"] >= min(yb))
            & (agg_df["y"] <= max(yb))
        )
        region_mean = np.abs(agg_df[mask]["peak_integral"]).mean()
        mean_charges[key] = region_mean

        x0 = np.searchsorted(x_vals, xb[0])
        x1 = np.searchsorted(x_vals, xb[1])
        y0 = np.searchsorted(y_vals, yb[0])
        y1 = np.searchsorted(y_vals, yb[1])

        rect = Rectangle(
            (x0, y0), x1 - x0, y1 - y0, linewidth=2, edgecolor="black", facecolor="none"
        )
        ax.add_patch(rect)

    plt.tight_layout()
    plt.show()

    return mean_charges

In [ ]:
mean_charges = plot_mean_charge_map(
    df_corrected, voltage, x_borders=x_borders, y_borders=y_borders
)